In [ ]:
import os
from itertools import product

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from dotenv import load_dotenv

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline

In [ ]:
# загружаем переменные окружения и объявляем константы

load_dotenv()

MFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
TRAIN_DATA = pd.read_csv("new_train_data.csv")
VAL_DATA = pd.read_csv("new_val_data.csv")
RANDOM_SEED = 42

Рассмотрим распределение таргета на данных
(если `synthetic `!= 0, то данные не являются полностью реальными)

In [729]:
TRAIN_DATA[["synthetic", "sentiment"]].value_counts()

synthetic  sentiment 
0          normal        6000
1          external      3675
0          external      2273
2          harassment    2000
1          threat         462
3          harassment     198
0          threat           2
Name: count, dtype: int64

In [730]:
VAL_DATA[["synthetic", "sentiment"]].value_counts()

synthetic  sentiment 
0          normal        8000
           external      2274
1          threat         100
0          harassment      87
           threat           2
Name: count, dtype: int64

Теперь переменная `VAL_DATA` содержит валидационный датасет, состоящий из реальных примером (за исключением класса `threat`, который из-за своей маленькости был расширен).
В переменной `TRAIN_DATA` содержатся все синтетические данные и по половине реальных сообщений для классов, отличных от `normal`

In [732]:
# обучим самую простую модель на этих данных и посмотрим на classification report

X_train, y_train = TRAIN_DATA["text"], TRAIN_DATA["sentiment"]
X_test, y_test = VAL_DATA["text"], VAL_DATA["sentiment"]

pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        ("logreg", LogisticRegression(random_state=RANDOM_SEED)),
    ]
)

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('logreg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [733]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    external       0.74      0.79      0.76      2274
  harassment       0.92      0.80      0.86        87
      normal       0.93      0.92      0.93      8000
      threat       0.79      0.63      0.70       102

    accuracy                           0.89     10463
   macro avg       0.85      0.78      0.81     10463
weighted avg       0.89      0.89      0.89     10463



# Improve baseline model

В этой части ноубука будет проведена серия экспериментов с архитектурами и параметрами. Среди них будет:
1. Подбор параметров для tf-idf + logistic regression
2. Эмбединги на основе предобученной bert модели + logistic regression
3. Эмбединги на основе предобученной bert модели + catboost
4. Дообучение классификатора на основе предобученной bert модели

Все эксперименты будут логироваться в *MLFlow* для удобного сравнения и анализа в будущем


In [ ]:
# подключаемся к серверу mlflow, где будут храниться метрики и информация об обучении модели

mlflow.set_tracking_uri(MFLOW_TRACKING_URI)
mlflow.set_experiment(experiment_id="1")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1783276282278, experiment_id='1', last_update_time=1783276282278, lifecycle_stage='active', name='v1', tags={}, trace_location=None, workspace='default'>

In [ ]:
def log_mlflow(
    model: Pipeline,
    model_type: str,
    metrics: dict,
    params: dict,
    artifacts: list[str],
    run_name: str,
) -> None:
    """
    Функция, логирующая параметры, метрики и артифакты в mlflow
    """
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        if artifacts:
            for artifact in artifacts:
                mlflow.log_artifact(artifact)

        if model_type == "sklearn":
            mlflow.sklearn.log_model(model, "model")

        elif model_type == "catboost":
            mlflow.catboost.log_model(model, "model")

In [ ]:
def train_tfidf_logreg(
    X_train: pd.DataFrame, y_train: pd.Series, tf_params: dict, lr_params: dict
) -> Pipeline:
    """
    Обучает и возвращает объект Pipeline. Текстовые поля преобразуются с помощью TfidfVectorizer
    На полученных эмбедингах обучается LogisticRegression
    """
    pipeline = Pipeline(
        [
            ("tfidf", TfidfVectorizer(**tf_params) if tf_params else TfidfVectorizer()),
            ("clf", LogisticRegression(**lr_params)),
        ]
    )

    pipeline.fit(X_train, y_train)
    return pipeline


def save_confusion_matrix_png(
    y_true: pd.Series, y_pred: np.ndarray, path_png: str
) -> str:
    """
    Строит матрицу ошибок, визуализирует её сохраняет в PNG и возвращает путь к файлу.
    """

    dir_name = os.path.dirname(path_png)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)

    classes = sorted(list(set(np.unique(y_true)) | set(np.unique(y_pred))))
    cm = confusion_matrix(y_true, y_pred, labels=classes)

    # Матрицу будут сохранять в .png при помощи функции heatmap из seaborn
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
        square=True,
    )
    plt.title("Confusion Matrix", fontsize=16, pad=20, fontweight="bold")
    plt.ylabel("Actual Labels", fontsize=12, labelpad=10)
    plt.xlabel("Predicted Labels", fontsize=12, labelpad=10)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(path_png, dpi=300)
    plt.close()

    return path_png


def get_metrics(y_test: pd.Series, y_pred: pd.Series) -> dict:
    """
    Возвращает словарь с основными метриками, среди которых precision,
    recall, f1, support для каждого класса в разрезе.
    """
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    class_metrics = {
        cls: {
            "precision": report[cls]["precision"],
            "recall": report[cls]["recall"],
            "f1": report[cls]["f1-score"],
            "support": report[cls]["support"],
        }
        for cls in report
        if cls not in ["accuracy", "macro avg", "weighted avg"]
    }

    metrics = {}
    for cls, vals in class_metrics.items():
        metrics[f"{cls}_precision"] = vals["precision"]
        metrics[f"{cls}_recall"] = vals["recall"]
        metrics[f"{cls}_f1"] = vals["f1"]
        metrics[f"{cls}_support"] = vals["support"]

    return metrics

In [ ]:
# tfidf параметры
max_features = [10000, None]
ngram_range = [(1, 5), (1, 1), (1, 3)]
max_df = [0.1, 0.05, 0.01]
analyzer = ["word", "char_wb"]

# lr параметры
C = [0.1, 1.0, 5.0, 0.5]
solver = ["saga", "lbfgs"]
max_iter = [1000]
class_weight = ["balanced", None]

# формируем сетку параметров
param_grid = {
    "max_features": max_features,
    "ngram_range": ngram_range,
    "max_df": max_df,
    "C": C,
    "solver": solver,
    "max_iter": max_iter,
    "class_weight": class_weight,
    "analyzer": analyzer,
}

In [ ]:
# перебираем все возможные комбинации параметров

for idx, params in enumerate(ParameterGrid(param_grid)):
    tf_params = {
        "max_features": params["max_features"],
        "ngram_range": params["ngram_range"],
        "max_df": params["max_df"],
        "analyzer": params["analyzer"],
    }

    lr_params = {
        "C": params["C"],
        "solver": params["solver"],
        "max_iter": params["max_iter"],
        "class_weight": params["class_weight"],
    }

    # train-test-split
    X_train, y_train = TRAIN_DATA["text"], TRAIN_DATA["sentiment"]
    X_test, y_test = VAL_DATA["text"], VAL_DATA["sentiment"]

    # обучаем tfidf + logreg и сохраняем метрики
    model = train_tfidf_logreg(X_train, y_train, tf_params, lr_params)

    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    os.makedirs("./temp_cms", exist_ok=True)
    train_path_png = f"./temp_cms/train_cm_run_{idx}.png"
    test_path_png = f"./temp_cms/test_cm_run_{idx}.png"

    # пути до матриц ошибок на трейне и тесте
    path_train_cm = save_confusion_matrix_png(
        y_true=y_train, y_pred=train_preds, path_png=train_path_png
    )
    path_test_cm = save_confusion_matrix_png(
        y_true=y_test, y_pred=test_preds, path_png=test_path_png
    )

    metrics = get_metrics(y_test, pd.Series(test_preds))

    log_mlflow(
        model=model,
        model_type="sklearn",
        metrics=metrics,
        params=tf_params | lr_params,
        artifacts=[
            path_train_cm,
            path_test_cm,
        ],  # матрицы ошибок будут в артефактах модели
        run_name=f"logreg_tfidf_run_{idx}",
    )

Результаты записаны на mlflow и будут участвовать в сравнении далее

# BERT + Logreg (Catboost) or STF

В этом блоке в качестве слоя-эмбедингов будут выступать разные предобученные bert модели. Попробуем и залогируем как логистические регрессии, обученные на эмбедингах, так и дообученные bert модели.
Первой будет следующая модель: https://huggingface.co/cointegrated/rubert-tiny-toxicity. Из ее описания на HF можно найти, что она обучалась на коротких текстах на русском языке для задачи классификации: 

```text
The problem is formulated as multilabel classification with the following classes:
   - non-toxic: the text does NOT contain insults, obscenities, and threats, in the sense of the OK ML Cup competition.
   - insult
   - obscenity
   - threat
   - dangerous: the text is inappropriate, in the sense of Babakov et.al., i.e. it can harm the reputation of the speaker.
```

In [ ]:
from catboost import CatBoostClassifier
from sentence_transformers import SentenceTransformer
from transformers import set_seed

set_seed(RANDOM_SEED)

In [287]:
model = SentenceTransformer("cointegrated/rubert-tiny-toxicity")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 2952.22it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny-toxicity
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def train_bert_logreg(
    X_train_embeddings: np.ndarray, y_train: pd.Series, lr_params: dict = None
) -> LogisticRegression:
    logreg = LogisticRegression(**lr_params)
    logreg.fit(X_train_embeddings, y_train)
    return logreg


def train_bert_catboost(
    X_train_embeddings: np.ndarray, y_train: pd.Series, cat_params: dict = None
) -> CatBoostClassifier:
    catboost = CatBoostClassifier(verbose=False, task_type="GPU", **cat_params)
    catboost.fit(X_train_embeddings, y_train)
    return catboost

## Logreg + bert

Тут пробуем на эмбедингах от bert обучить логистическую регрессию

In [ ]:
mlflow.set_experiment(experiment_id="2")

In [ ]:
# формируем сетку параметров (все те же что и для логрега, но без tfidf)
C = [0.1, 1.0, 10.0, 0.5]
solver = ["saga", "lbfgs"]
max_iter = [1000, 3000]
class_weight = ["balanced"]

lr_grid = {
    "C": C,
    "solver": solver,
    "max_iter": max_iter,
    "class_weight": class_weight,
}

In [ ]:
# тот же цикл обучения, но без параметров для tf-idf. сохраняем те же метрики и артефакты
# сразу сделаем эмбединги из текстовых колонок и будем обучать

X_train, y_train = TRAIN_DATA["text"], TRAIN_DATA["sentiment"]
X_test, y_test = VAL_DATA["text"], VAL_DATA["sentiment"]

X_train_embeddings = model.encode(X_train.to_list())
X_test_embeddings = model.encode(X_test.to_list())

# перебираем все возможные сочетания параметров
for idx, params in enumerate(ParameterGrid(lr_grid)):
    lr_params = {
        "C": params["C"],
        "solver": params["solver"],
        "max_iter": params["max_iter"],
        "class_weight": params["class_weight"],
    }

    model_logreg_bert = train_bert_logreg(X_train_embeddings, y_train, lr_params)

    train_preds = model_logreg_bert.predict(X_train_embeddings)
    test_preds = model_logreg_bert.predict(X_test_embeddings)

    os.makedirs("./temp_cms", exist_ok=True)
    train_path_png = f"./temp_cms/train_cm_run_{idx}.png"
    test_path_png = f"./temp_cms/test_cm_run_{idx}.png"
    path_train_cm = save_confusion_matrix_png(
        y_true=y_train, y_pred=train_preds, path_png=train_path_png
    )
    path_test_cm = save_confusion_matrix_png(
        y_true=y_test, y_pred=test_preds, path_png=test_path_png
    )

    metrics = get_metrics(y_test, pd.Series(test_preds))

    log_mlflow(
        model=model,
        model_type="sklearn",
        metrics=metrics,
        params=lr_params,
        artifacts=[path_train_cm, path_test_cm],
        run_name=f"logreg_bert_run_{idx}",
    )

## Catboost

Тут пробуем обучить градиентный бустинг (на примере catboost) используя его способности из коробки

In [54]:
mlflow.set_experiment(experiment_id="4")

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1780935174121, experiment_id='4', last_update_time=1780935174121, lifecycle_stage='active', name='bert_catboost', tags={}, trace_location=None, workspace='default'>

In [291]:
# Формируем сетку параметров для многоклассового CatBoost
iterations = [500, 1000]
learning_rate = [0.03, 0.1, 0.2]
depth = [4, 6, 8]
l2_leaf_reg = [1, 3, 5, 10]
loss_function = ["MultiClass"]
auto_class_weights = ["Balanced"]

cb_grid = {
    "iterations": iterations,
    "learning_rate": learning_rate,
    "depth": depth,
    "l2_leaf_reg": l2_leaf_reg,
    "loss_function": loss_function,
    "auto_class_weights": auto_class_weights,
}

In [ ]:
# аналогичный блок обучения, что и был для логистической регрессии

X_train, y_train = TRAIN_DATA["text"], TRAIN_DATA["sentiment"]
X_test, y_test = VAL_DATA["text"], VAL_DATA["sentiment"]


for idx, params in enumerate(ParameterGrid(cb_grid)):
    model_catboost_bert = train_bert_catboost(X_train_embeddings, y_train, params)

    train_preds = model_catboost_bert.predict(X_train_embeddings)
    test_preds = model_catboost_bert.predict(X_test_embeddings)

    train_preds_flat = train_preds.flatten()
    test_preds_flat = test_preds.flatten()

    os.makedirs("./temp_cms", exist_ok=True)
    train_path_png = f"./temp_cms/train_cm_run_{idx}.png"
    test_path_png = f"./temp_cms/test_cm_run_{idx}.png"

    path_train_cm = save_confusion_matrix_png(
        y_true=y_train, y_pred=train_preds_flat, path_png=train_path_png
    )
    path_test_cm = save_confusion_matrix_png(
        y_true=y_test, y_pred=test_preds_flat, path_png=test_path_png
    )

    metrics = get_metrics(y_test, pd.Series(test_preds_flat))

    log_mlflow(
        model=model_catboost_bert,
        model_type="catboost",
        metrics=metrics,
        params=params,
        artifacts=[path_train_cm, path_test_cm],
        run_name=f"catboost_bert_run_{idx}",
    )

## BERT classifier 

In [ ]:
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

In [ ]:
def log_mlflow_bert(path_to_bert_model, metrics, artifacts, run_name):
    model = AutoModelForSequenceClassification.from_pretrained(path_to_bert_model)
    tokenizer = AutoTokenizer.from_pretrained(path_to_bert_model)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_metrics(metrics)
        if artifacts:
            for artifact in artifacts:
                mlflow.log_artifact(artifact)

        mlflow.transformers.log_model(
            transformers_model={"model": model, "tokenizer": tokenizer},
            artifact_path="bert_model",
            task="text-classification",
        )

### rubert-tiny-toxicity

В начале попробуем дообучить тот же самый bert, который был использован ранее для получения эмбедингов

In [ ]:
model_name = "cointegrated/rubert-tiny-toxicity"
number_of_labels = 4  # normal, external, harassment, threat

label_to_number = {
    label: i for i, label in enumerate(sorted(TRAIN_DATA["sentiment"].unique()))
}
number_to_label = {i: label for label, i in label_to_number.items()}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=number_of_labels,
    id2label=number_to_label,
    label2id=label_to_number,
    ignore_mismatched_sizes=True,
    problem_type="single_label_classification",
)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 57/57 [00:00<00:00, 3337.64it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny-toxicity
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([5]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([5, 312]) vs model:torch.Size([4, 312])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
def tokenize_function(examples: pd.Series) -> list[int]:
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=512
    )


def compute_metrics(eval_pred: np.array) -> dict:
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    y_test_series = pd.Series(labels)
    y_pred_series = pd.Series(predictions)
    calculated_metrics = get_metrics(y_test_series, y_pred_series)

    # сумма f1 скоров нужна лишь для того, чтобы при валидации trainer знал, как выбрать лучшую модель
    # чтобы код работал корректно. ориентироваться мы будем на остальные метрики отдельно, сохраняться модель будет каждую эпоху
    f1_values = [
        value for key, value in calculated_metrics.items() if key.endswith("_f1")
    ]
    calculated_metrics["sum_f1"] = sum(f1_values)

    return calculated_metrics

In [ ]:
X_train, y_train = (
    TRAIN_DATA["text"],
    TRAIN_DATA["sentiment"].map(label_to_number),
)  # мапим классы на цифры

eval_dataset = Dataset.from_pandas(
    pd.DataFrame(
        {"text": VAL_DATA["text"], "label": VAL_DATA["sentiment"].map(label_to_number)}
    )
)
train_dataset = Dataset.from_pandas(pd.DataFrame({"text": X_train, "label": y_train}))

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 10463/10463 [00:01<00:00, 9069.77 examples/s]


Для того, чтобы обучение шло лучше, необходимо поработать с дисбалансом классом. Для этого будем использоваться взвешенный лосс.
Нативно объект trainer не имеет такого выбора, поэтому обернем его в класс `WeightedTrainer` с модифицированным лоссом

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)
device = "cuda"
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Стандартный CrossEntropyLoss из torch может быть модифицирован
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [410]:
training_args = TrainingArguments(
    output_dir="./bert_logreg_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    logging_dir="./bert_logs",
    logging_steps=250,
    learning_rate=2e-5,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    seed=RANDOM_SEED,
    data_seed=RANDOM_SEED,
    metric_for_best_model="sum_f1",
    greater_is_better=True,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [411]:
number_to_label

{0: 'external', 1: 'harassment', 2: 'normal', 3: 'threat'}

In [412]:
trainer.train()

Epoch,Training Loss,Validation Loss,0 Precision,0 Recall,0 F1,0 Support,1 Precision,1 Recall,1 F1,1 Support,2 Precision,2 Recall,2 F1,2 Support,3 Precision,3 Recall,3 F1,3 Support,Sum F1
1,0.282642,0.198734,0.845577,0.992084,0.912991,2274.000000,0.690909,0.873563,0.771574,87.000000,0.996831,0.943750,0.969565,8000.000000,0.873874,0.950980,0.910798,102.000000,3.564927
2,0.279306,0.180052,0.865562,0.988127,0.922793,2274.000000,0.652893,0.908046,0.759615,87.000000,0.995945,0.951750,0.973346,8000.000000,0.960396,0.950980,0.955665,102.000000,3.611419
3,0.206132,0.192384,0.858896,0.985048,0.917657,2274.000000,0.707547,0.862069,0.777202,87.000000,0.994764,0.949875,0.971801,8000.000000,0.900000,0.970588,0.933962,102.000000,3.600622
4,0.203681,0.184129,0.879173,0.972735,0.923591,2274.000000,0.638655,0.873563,0.737864,87.000000,0.991189,0.956250,0.973406,8000.000000,0.900000,0.970588,0.933962,102.000000,3.568823
5,0.223686,0.188051,0.878512,0.976253,0.924807,2274.000000,0.672566,0.873563,0.760000,87.000000,0.992220,0.956500,0.974033,8000.000000,0.891892,0.970588,0.929577,102.000000,3.588417


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 31.39it/s]


TrainOutput(global_step=9135, training_loss=0.2529325582319488, metrics={'train_runtime': 1092.8284, 'train_samples_per_second': 66.845, 'train_steps_per_second': 8.359, 'total_flos': 538826300006400.0, 'train_loss': 0.2529325582319488, 'epoch': 5.0})

In [ ]:
# берем чекпоинт после второй эпохи, так как после 3й эпохи модель начала переобучаться и метрики на валидации стали падать

trainer.model = AutoModelForSequenceClassification.from_pretrained(
    "bert_logreg_model/checkpoint-5481"
).to(device)

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 4300.53it/s]


In [ ]:
# загрузим метрики на трейне и тесте в mlflow

logits_test = trainer.predict(tokenized_eval_dataset).predictions
logits_train = trainer.predict(tokenized_train_dataset).predictions

predictions_test = np.argmax(logits_test, axis=-1)
predictions_train = np.argmax(logits_train, axis=-1)

y_test_series = pd.Series(VAL_DATA["sentiment"].map(label_to_number))
y_train_series = pd.Series(TRAIN_DATA["sentiment"].map(label_to_number))

y_pred_series = pd.Series(predictions_test)
calculated_metrics_bert = get_metrics(y_test_series, y_pred_series)
path_train_cm_bert = save_confusion_matrix_png(
    y_true=y_train_series, y_pred=predictions_train, path_png="model/temp_cms"
)
path_test_cm_bert = save_confusion_matrix_png(
    y_true=y_test_series, y_pred=predictions_test, path_png="model/temp_cms"
)

# Find best model

In [ ]:
# лучшая логистическая регрессия

pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char_wb", ngram_range=(1, 5), max_features=10000, max_df=0.05
            ),
        ),
        (
            "clf",
            LogisticRegression(
                random_state=RANDOM_SEED,
                max_iter=1000,
                class_weight={
                    "external": 0.25,
                    "normal": 1,
                    "harassment": 0.25,
                    "threat": 1,
                },
            ),
        ),
    ]
)

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [ ]:
print(classification_report(y_test, pipeline.predict(X_test)))

              precision    recall  f1-score   support

    external       0.94      0.80      0.86      2274
  harassment       0.92      0.63      0.75        87
      normal       0.94      0.98      0.96      8000
      threat       0.88      0.66      0.75       102

    accuracy                           0.94     10463
   macro avg       0.92      0.77      0.83     10463
weighted avg       0.94      0.94      0.94     10463



In [ ]:
# лучшая бертовская модель

print(
    classification_report(
        y_test,
        [
            number_to_label[p]
            for p in trainer.predict(tokenized_eval_dataset).predictions.argmax(axis=1)
        ],
    )
)

              precision    recall  f1-score   support

    external       0.86      0.99      0.92      2274
  harassment       0.71      0.86      0.78        87
      normal       0.99      0.95      0.97      8000
      threat       0.90      0.97      0.93       102

    accuracy                           0.96     10463
   macro avg       0.87      0.94      0.90     10463
weighted avg       0.96      0.96      0.96     10463



In [ ]:
# получаем логиты и вероятности для теста и трейна

bert_logits_train = trainer.predict(tokenized_train_dataset).predictions
bert_probs_train = torch.softmax(torch.tensor(bert_logits_train), dim=1).numpy()

bert_logits_test = trainer.predict(tokenized_eval_dataset).predictions
bert_probs_test = torch.softmax(torch.tensor(bert_logits_test), dim=1).numpy()

In [704]:
logreg_probs_train = pipeline.predict_proba(X_train)
logreg_probs_test = pipeline.predict_proba(X_test)

## Threshold grid search

Мы получили две модели, каждая из которых удовлетворительно находит какие-то классы. Берт имеет высокую точность на нормальных сообщений и сообщениях, содержащих угрозы, логрег - на всех остальных.
Для нужной точности будем использовать оба предсказания (вернее распределения вероятностей) с весами, которые подберем сеткой. Предсказания делаются по формуле:

$$(1 - w_k) * predictions_{logreg} + w_k * predictions_{bert}$$

$$k = 1,2,3,4$$

In [ ]:
def decision_rule(bert_probs: np.ndarray, logreg_probs: np.ndarray, weights: dict):
    n_classes = bert_probs.shape[1]
    probs = np.zeros_like(bert_probs)
    for k in range(n_classes):
        w_b = weights[k]
        probs[:, k] = w_b * bert_probs[:, k] + (1 - w_b) * logreg_probs[:, k]

    return [number_to_label[x] for x in np.argmax(probs, axis=1)]

In [ ]:
best_score = -1
best_params = None

for w_har, w_norm, w_thr, w_ext in product(
    np.arange(0.0, 1.01, 0.1),
    np.arange(0.0, 1.01, 0.1),
    np.arange(0.0, 1.01, 0.1),
    np.arange(0.0, 1.01, 0.1),
):
    weights = [w_har, w_norm, w_thr, w_ext]
    preds = decision_rule(bert_probs_test, logreg_probs_test, weights)
    report = classification_report(y_test, preds, output_dict=True)

    # берем минимальный precision по классам в качестве метрики
    score = min(
        report["harassment"]["precision"],
        report["threat"]["precision"],
        report["external"]["precision"],
    )

    if score > best_score:
        best_score = score
        best_params = (w_har, w_norm, w_thr, w_ext)

In [ ]:
# финальные метрики на тесте с найденными весами

preds = decision_rule(
    bert_probs_test, logreg_probs_test, {0: 0.1, 1: 0.2, 2: 0, 3: 0.6}
)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

    external       0.94      0.82      0.88      2274
  harassment       0.97      0.69      0.81        87
      normal       0.95      0.98      0.97      8000
      threat       0.96      0.87      0.91       102

    accuracy                           0.95     10463
   macro avg       0.95      0.84      0.89     10463
weighted avg       0.95      0.95      0.94     10463



In [ ]:
# сохраняем пайплайн логрегрессии и модель берта с токенизатором

joblib.dump(pipeline, "logreg_pipeline.pkl")
model.save_pretrained("bert_tiny_ens")
tokenizer.save_pretrained("bert_tiny_ens")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.43it/s]


('bert_tiny_ens/tokenizer_config.json', 'bert_tiny_ens/tokenizer.json')